# Report figure generation

Generates the Section Two term-structure and survival-curve figures using the shared Matplotlib style helpers.


In [ ]:
import numpy as np 
import matplotlib.pyplot as plt 
from fig_style import use_ieee_style, new_figure, save_figure
from matplotlib.ticker import FormatStrFormatter


use_ieee_style(single_column=False, height_factor=0.9, use_tex=True)

# Short rate parameters
a_r = 0.02
b_r = 0.02
c_r = 0.35

# Hazard rate parameters
# IG
a_h_ig = 0.015
b_h_ig = -0.012
c_h_ig = 0.30

# HY
a_h_hy = 0.03
b_h_hy = -0.02
c_h_hy = 0.40

R = 0.4 # Recovery Rate

r = lambda t: a_r + b_r * np.exp(-c_r * t)              

h_ig = lambda t: a_h_ig + b_h_ig * np.exp(-c_h_ig * t)  
h_hy = lambda t: a_h_hy + b_h_hy * np.exp(-c_h_hy * t)  

T = np.linspace(0.0, 10.0, 500) 

r_T = r(T)
h_ig_T = h_ig(T)
h_hy_T = h_hy(T)

D0    = lambda t : np.exp(-a_r*t    - (b_r/c_r)      *(1-np.exp(-c_r  *t))  )
S0_ig = lambda t : np.exp(-a_h_ig*t - (b_h_ig/c_h_ig)*(1-np.exp(-c_h_ig*t)) )
S0_hy = lambda t : np.exp(-a_h_hy*t - (b_h_hy/c_h_hy)*(1-np.exp(-c_h_hy*t)) )

S_ig = S0_ig(T)
S_hy = S0_hy(T)
D0T = D0(T)


fig, axes = new_figure(2, 2, single_column=False, height_factor=0.9)

# 1,1 Short rate
ax = axes[0, 0]
ax.plot(T, r_T, linewidth=2)
ax.set_title("Short rate $r(t)$")
ax.set_xlabel("Time (years)")
ax.set_ylabel("Rate")
ax.set_xlim(T.min(), T.max()) # X axis tight
ax.yaxis.set_major_formatter(FormatStrFormatter('%.3f'))


# 1,2 Hazard rates IG and HY
ax = axes[0, 1]
ax.plot(T, h_ig_T, label="IG $h(t)$", linewidth=2)
ax.plot(T, h_hy_T, label="HY $h(t)$", linewidth=2, linestyle="--")
ax.set_title("Hazard rate $h(t)$")
ax.set_xlabel("Time (years)")
ax.set_ylabel("Intensity")
ax.set_xlim(T.min(), T.max()) # X axis tight
ax.legend(loc="best", frameon=True)
ax.yaxis.set_major_formatter(FormatStrFormatter('%.3f'))

# 2,1 Discount factor 
ax = axes[1, 0]
ax.plot(T, D0T, linewidth=2)
ax.set_title("Discount factor $D(0, t)$")
ax.set_xlabel("Time (years)")
ax.set_ylabel("$D(0, t)$")
ax.set_xlim(T.min(), T.max()) # X axis tight

# 2,2 Survival probabilities IG and HY
ax = axes[1, 1]
ax.plot(T, S_ig, label="IG $S(t)$", linewidth=2)
ax.plot(T, S_hy, label="HY $S(t)$", linewidth=2, linestyle="--")
ax.set_title("Survival probability $S(0,t)$")
ax.set_xlabel("Time (years)")
ax.set_ylabel("$S(0,t)$")

ax.set_xlim(T.min(), T.max()) # X axis tight
ax.legend(loc="best", frameon=True)

save_figure(fig, "term-structures-2x2.pdf")
plt.show() 

# Credit spreads in decimal
s_ig = lambda t: (1 - R) * h_ig(t)
s_hy = lambda t: (1 - R) * h_hy(t)

bps = lambda x: 1e4 * x   # 1 basis point = 0.0001

fig, ax = new_figure(single_column=True, height_factor=0.75)

ax.plot(T, bps(s_ig(T)), label="IG spread", linewidth=1.0, color="black", linestyle="-")
ax.plot(T, bps(s_hy(T)), label="HY spread", linewidth=1.0, color="black", linestyle="--")

ax.set_title("Investment grade vs high yield credit spreads")
ax.set_xlabel("Time (years)")
ax.set_ylabel("Spread (bps)")
# ax.grid(True, which="both", alpha=0.3)
ax.legend(loc="best", frameon=True)

# X axis tight
ax.set_xlim(T.min(), T.max())

# Y limits
y_data_min = min(bps(s_ig(T)).min(), bps(s_hy(T)).min())
y_data_max = max(bps(s_ig(T)).max(), bps(s_hy(T)).max())

# Ensure zero is in view
y_min = min(0, y_data_min)

# Cap upper limit at 190
y_max = max(190, y_data_max)

ax.set_ylim(y_min, y_max)

save_figure(fig, "credit-spreads-bps.pdf")
plt.show()